# MaizeScan — Training Notebook v1
## Binary CNN: Healthy vs. Diseased Maize Leaves (MobileNetV2)

### Before you start
1. **Enable GPU:** Runtime → Change runtime type → Hardware accelerator: **T4 GPU**
2. **Kaggle token:** [kaggle.com → Settings → API → Create New Token](https://www.kaggle.com/settings) — downloads `kaggle.json`
3. **Google Drive** will be mounted at `/content/drive` for data + artifact persistence

### Estimated runtime
| Step | Time |
|---|---|
| Environment setup | ~5 min |
| Dataset download (2.3 GB) | ~10 min |
| Phase 1 training (frozen base) | ~10 min |
| Phase 2 fine-tuning | ~35 min |
| Export + benchmark | ~5 min |
| **Total** | **~65 min** |

### Output artifacts
- `mobilenetv2_best.tflite` — INT8 quantized model for the FastAPI backend (~3.5 MB)
- `mobilenetv2_meta.json` — metrics + version metadata read by `/health` and `/model/info`

Both are saved to Google Drive **and** downloaded to your machine at the end.

In [ ]:
# ── Cell 1: Environment Setup ─────────────────────────────────────────────────
import subprocess, sys, os, shutil, json, random, logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# Install pinned TF version (matches production Dockerfile.api)
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'tensorflow==2.15.0', 'scikit-learn', 'matplotlib',
    'seaborn', 'Pillow', 'numpy', 'kaggle', '-q'
], check=True)

# Mount Google Drive for data + artifact persistence across sessions
from google.colab import drive
drive.mount('/content/drive')

# ── Paths ──────────────────────────────────────────────────────────────────────
REPO_URL   = 'https://github.com/CBahtaria/maize-leaf-classifier.git'
REPO_DIR   = Path('/content/maize-leaf-classifier')
DATA_DIR   = Path('/content/data/maize')
DRIVE_BASE = Path('/content/drive/MyDrive/maize-classifier')
OUTPUT_DIR = DRIVE_BASE / 'training_output'
ARTIFACTS  = DRIVE_BASE / 'model_artifacts'

for p in [DATA_DIR, OUTPUT_DIR, ARTIFACTS]:
    p.mkdir(parents=True, exist_ok=True)

# Clone repo so we can import model modules
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('\n✓ Environment ready')
print(f'  Repo:      {REPO_DIR}')
print(f'  Data:      {DATA_DIR}')
print(f'  Drive out: {ARTIFACTS}')

In [ ]:
# ── Cell 2: Kaggle Credentials + Dataset Download ─────────────────────────────
# First time: upload kaggle.json (file picker opens below)
# Subsequent sessions: cached copy in Drive is reused automatically

from google.colab import files as colab_files

KAGGLE_CACHE = DRIVE_BASE / 'kaggle.json'

if KAGGLE_CACHE.exists():
    print('Using cached kaggle.json from Drive')
    kaggle_src = KAGGLE_CACHE
else:
    print('Upload your kaggle.json (kaggle.com → Settings → API → Create New Token):')
    uploaded = colab_files.upload()
    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError('kaggle.json not uploaded — check the file name')
    kaggle_src = Path('/content/kaggle.json')
    shutil.copy(kaggle_src, KAGGLE_CACHE)
    print(f'✓ Saved to Drive for reuse: {KAGGLE_CACHE}')

kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(exist_ok=True)
shutil.copy(kaggle_src, kaggle_dir / 'kaggle.json')
os.chmod(kaggle_dir / 'kaggle.json', 0o600)
print('✓ Kaggle credentials configured')

# Download PlantVillage (~2.3 GB unzipped) — skip if raw dir already present
RAW_DIR = Path('/content/raw')
if not RAW_DIR.exists():
    print('Downloading PlantVillage dataset (~2.3 GB) …')
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', 'abdallahalidev/plantvillage-dataset',
        '-p', '/content/raw', '--unzip'
    ], check=True)
    print('✓ Download complete')
else:
    print('✓ Raw dataset already present — skipping download')

# Locate maize class folders inside the colour subset
pv_color = RAW_DIR / 'plantvillage dataset' / 'color'
if not pv_color.exists():
    # Some versions unzip without the space
    alt = RAW_DIR / 'plantvillage_dataset' / 'color'
    pv_color = alt if alt.exists() else pv_color

exts = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}
maize_classes = sorted(
    d for d in pv_color.iterdir()
    if d.is_dir() and d.name.lower().startswith('corn_(maize)')
)
print(f'\nFound {len(maize_classes)} maize class folders:')
for c in maize_classes:
    n = sum(1 for f in c.iterdir() if f.suffix in exts)
    kind = 'healthy' if 'healthy' in c.name.lower() else 'diseased'
    print(f'  [{kind}] {c.name} ({n} images)')

# Copy to DATA_DIR preserving original folder names.
# model/preprocess.py binary_map(): "healthy" → 0, anything else → 1
total_copied = 0
for cls_dir in maize_classes:
    dest = DATA_DIR / cls_dir.name
    if dest.exists() and any(dest.iterdir()):
        print(f'  {cls_dir.name}: already present, skipping')
        continue
    dest.mkdir(parents=True, exist_ok=True)
    imgs = [f for f in cls_dir.iterdir() if f.suffix in exts]
    for f in imgs:
        shutil.copy(f, dest / f.name)
    total_copied += len(imgs)
    print(f'  Copied {len(imgs):,} → {dest.name}')

print(f'\n✓ {total_copied:,} new images copied to {DATA_DIR}')

In [ ]:
# ── Cell 3: Dataset Inspection ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from model.preprocess import scan_dataset

paths, labels = scan_dataset(DATA_DIR)
healthy_p  = [p for p, l in zip(paths, labels) if l == 0]
diseased_p = [p for p, l in zip(paths, labels) if l == 1]

print(f'Total images : {len(paths):,}')
print(f'Healthy      : {len(healthy_p):,}  ({len(healthy_p)/len(paths)*100:.1f}%)')
print(f'Diseased     : {len(diseased_p):,}  ({len(diseased_p)/len(paths)*100:.1f}%)')

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('PlantVillage — Maize Leaf Samples', fontsize=13, fontweight='bold')
for ax in axes[0]:
    img = Image.open(random.choice(healthy_p)).resize((224, 224))
    ax.imshow(img); ax.axis('off')
    ax.set_title('Healthy', color='#1B7A3E', fontsize=9, fontweight='bold')
for ax in axes[1]:
    img = Image.open(random.choice(diseased_p)).resize((224, 224))
    ax.imshow(img); ax.axis('off')
    ax.set_title('Diseased', color='#C0392B', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig(str(ARTIFACTS / 'sample_images.png'), dpi=100, bbox_inches='tight')
plt.show()
print('✓ Sample grid saved to Drive')

In [ ]:
# ── Cell 4: Stratified Split + Class Weights ──────────────────────────────────
from model.preprocess import stratified_split, compute_class_weights

(train_p, train_l), (val_p, val_l), (test_p, test_l) = stratified_split(paths, labels)
class_weights = compute_class_weights(train_l)

print('70 / 15 / 15 stratified split:')
print(f'  Train : {len(train_p):>5,}  ({train_l.count(0):,} healthy / {train_l.count(1):,} diseased)')
print(f'  Val   : {len(val_p):>5,}  ({val_l.count(0):,} healthy / {val_l.count(1):,} diseased)')
print(f'  Test  : {len(test_p):>5,}  ({test_l.count(0):,} healthy / {test_l.count(1):,} diseased)')
print(f'\nClass weights (inverse-frequency, paper Section 3.7):')
print(f'  healthy={class_weights[0]:.4f}  diseased={class_weights[1]:.4f}')

In [ ]:
# ── Cell 5: Build Datasets + Train MobileNetV2 ────────────────────────────────
# Expected runtime: ~45 min on T4 GPU
# Progress is saved to Drive after each phase — safe to disconnect and resume

from model.preprocess import build_tf_dataset
from model.architectures import ARCH_REGISTRY
from model.train import train as train_model

ARCH = 'mobilenetv2'
preprocess_fn = ARCH_REGISTRY[ARCH]['preprocess']

print('Building tf.data pipelines…')
train_ds = build_tf_dataset(train_p, train_l, preprocess_fn, augment=True)
val_ds   = build_tf_dataset(val_p,   val_l,   preprocess_fn, augment=False)
test_ds  = build_tf_dataset(test_p,  test_l,  preprocess_fn, augment=False)
print('✓ Datasets ready')

# Representative images for INT8 quantization calibration (FIX-3)
rep_sample = random.sample(train_p, min(200, len(train_p)))
rep_images = np.stack([
    np.array(Image.open(p).resize((224, 224)), dtype=np.float32)
    for p in rep_sample
])
print(f'✓ Calibration images: {rep_images.shape}  dtype={rep_images.dtype}')
print(f'\nStarting training — output dir: {OUTPUT_DIR}\n')

results = train_model(
    arch_name=ARCH,
    train_ds=train_ds,
    val_ds=val_ds,
    test_ds=test_ds,
    class_weights=class_weights,
    output_dir=str(OUTPUT_DIR),
    representative_images=rep_images,
)
print('\n✓ Training complete — checkpoints saved to Drive')

In [ ]:
# ── Cell 6: Evaluation Metrics + Plots ────────────────────────────────────────
import tensorflow as tf
from IPython.display import Image as IPImage, display
from model.evaluate import generate_plots

m  = results['metrics']
ci = m['wilson_ci_95']

print('=' * 55)
print(f'  Architecture : MobileNetV2 (binary classifier)')
print(f'  Test samples : {m["n_test"]:,}')
print('=' * 55)
print(f'  Accuracy     : {m["accuracy"]:.4f}  [{ci[0]:.3f}–{ci[1]:.3f}] 95% CI')
print(f'  Precision    : {m["precision"]:.4f}')
print(f'  Sensitivity  : {m["sensitivity"]:.4f}  (Recall / TPR)')
print(f'  Specificity  : {m["specificity"]:.4f}  (TNR)')
print(f'  F1 Score     : {m["f1"]:.4f}')
print(f'  AUC-ROC      : {m["auc_roc"]:.4f}')
print('=' * 55)
print(f'  TP={m["tp"]}  TN={m["tn"]}  FP={m["fp"]}  FN={m["fn"]}')
print('=' * 55)

# Regenerate predictions for plots
model = tf.keras.models.load_model(results['model_path'])
y_true_all, y_scores_all = [], []
test_ds_eval = build_tf_dataset(test_p, test_l, preprocess_fn, augment=False)
for imgs, lbls in test_ds_eval:
    preds = model.predict(imgs, verbose=0)
    y_scores_all.extend(preds.flatten().tolist())
    y_true_all.extend(lbls.numpy().tolist())

y_true_np   = np.array(y_true_all, dtype=int)
y_scores_np = np.array(y_scores_all, dtype=float)
y_pred_np   = (y_scores_np >= 0.5).astype(int)

generate_plots(
    y_true=y_true_np, y_pred=y_pred_np, y_scores=y_scores_np,
    history={'phase1': results['phase1_history'], 'phase2': results['phase2_history']},
    output_dir=str(ARTIFACTS),
    arch_name='mobilenetv2',
)

for fname in ['mobilenetv2_confusion_matrix.png',
              'mobilenetv2_roc_curve.png',
              'mobilenetv2_training_history.png']:
    fpath = ARTIFACTS / fname
    if fpath.exists():
        print(f'\n{fname}')
        display(IPImage(str(fpath), width=500))

print('\n✓ Evaluation plots saved to Drive')

In [ ]:
# ── Cell 7: Rename Artifacts + Benchmark TFLite ───────────────────────────────
# The API reads MODEL_PATH=model_artifacts/mobilenetv2_best.tflite
# train() saves as mobilenetv2_int8.tflite — rename here.

from model.evaluate import measure_inference_time_tflite

src_tflite = Path(results['tflite_path'])               # mobilenetv2_int8.tflite
src_meta   = Path(OUTPUT_DIR) / 'mobilenetv2_meta.json'

best_tflite = ARTIFACTS / 'mobilenetv2_best.tflite'
best_meta   = ARTIFACTS / 'mobilenetv2_meta.json'

shutil.copy(src_tflite, best_tflite)
shutil.copy(src_meta,   best_meta)

size_mb = best_tflite.stat().st_size / 1e6
print(f'TFLite model: {size_mb:.2f} MB  (≈{int(size_mb*1024)} KB)')

# Benchmark on 50 random test images
bench_paths = random.sample(test_p, min(50, len(test_p)))
bench_imgs  = np.stack([
    np.array(Image.open(p).resize((224, 224)), dtype=np.float32)
    for p in bench_paths
])
inf_ms = measure_inference_time_tflite(str(best_tflite), bench_imgs, n_runs=50)
print(f'TFLite CPU inference: {inf_ms:.1f} ms / image  (n=50)')

# Update meta.json with deployment fields
meta = json.loads(best_meta.read_text())
meta.update({
    'tflite_path': str(best_tflite),
    'tflite_size_mb': round(size_mb, 3),
    'tflite_inference_ms': round(inf_ms, 1),
    'version': '1.0.0',
})
best_meta.write_text(json.dumps(meta, indent=2))

print(f'\nArtifacts in Drive:')
print(f'  {best_tflite}')
print(f'  {best_meta}')
print('✓ Ready to download')

In [ ]:
# ── Cell 8: Download Artifacts ────────────────────────────────────────────────
# Downloads both files to your machine.
# After downloading:
#   mkdir -p model_artifacts
#   mv ~/Downloads/mobilenetv2_best.tflite  model_artifacts/
#   mv ~/Downloads/mobilenetv2_meta.json    model_artifacts/
#
# Test locally (Docker dev env):
#   docker compose -f docker/docker-compose.dev.yml up --build
#   curl localhost:8000/health
#
# Deploy to VPS:
#   bash scripts/upload-model.sh deploy@<VPS_IP>

from google.colab import files as colab_files

print('Downloading mobilenetv2_best.tflite…')
colab_files.download(str(best_tflite))

print('Downloading mobilenetv2_meta.json…')
colab_files.download(str(best_meta))

print("""
╭────────────────────────────────────────────────────────────╮
│  Training complete!                                       │
│                                                           │
│  Downloaded:                                              │
│    mobilenetv2_best.tflite  (INT8 quantized, ~3.5 MB)     │
│    mobilenetv2_meta.json    (metrics + version info)      │
│                                                           │
│  Move both files to local model_artifacts/ then:          │
│    docker compose -f docker/docker-compose.dev.yml \\      │
│      up --build                                           │
│    curl localhost:8000/health  → {"status":"ok"}           │
╰────────────────────────────────────────────────────────────╯
""")

## Research Inconsistencies Fixed in This Implementation

| Fix | Paper Section | Issue | Resolution |
|-----|--------------|-------|------------|
| FIX-1 | 3.5.3 | `/255` normalization applied to all architectures — wrong for MobileNetV2, Xception, InceptionV3 | Architecture-specific `preprocess_input` embedded as Keras Lambda layer; model accepts raw `[0,255]` input |
| FIX-2 | 3.6.1 | Deprecated `ImageDataGenerator` used for augmentation | `tf.data.Dataset` + `tf.keras.layers.RandomFlip/Rotation/Zoom/Brightness` |
| FIX-3 | 3.12.2 | Paper claims mobile deployment but never specifies TFLite conversion | `export_tflite()` with full INT8 post-training quantization (~3.5 MB vs ~14 MB) |
| FIX-4 | 3.10.2 | Linear LR warmup described in paper but no Keras callback exists natively | `LinearWarmupCallback` implemented in `model/callbacks.py` |
| FIX-5 | Table 3.7 | Hardcoded layer indices for fine-tuning break across TF versions | Dynamic `len(base_model.layers) - fine_tune_n` in `architectures.py:unfreeze_top_n()` |
| FIX-6 | — | Paper URL typo: "colad.research.google.com" | Corrected to "colab.research.google.com" in all documentation |
| FIX-7 | 3.12.2 | Paper benchmarks `.h5` CPU inference but claims mobile deployment | Both `.keras` CPU benchmark (academic) and TFLite CPU benchmark (deployment-realistic) reported |